In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Audio
from tqdm import tqdm
import soundfile as sf
import librosa
from huggingface_hub import login

# ---------------- CONFIG ----------------
DATASET_NAME = "ai4bharat/IndicVoices"
LANG = "hindi"
OUT_DIR = "benchmark_data"
TARGET_SAMPLE_RATE = 16000

# Updated Buckets to be slightly more flexible
BUCKETS = {
    "5s": (4, 8),
    "15s": (12, 20),
    "30s": (20, 35)
}
CLIPS_PER_BUCKET = 10

# ----------------------------------------

def prepare_folders():
    for name in BUCKETS:
        os.makedirs(os.path.join(OUT_DIR, name), exist_ok=True)

def get_audio_data(token):
    print(f"Streaming {DATASET_NAME} ({LANG})...")
    login(token)

    # 1. Load and 2. Cast the column immediately
    ds = load_dataset(DATASET_NAME, LANG, split="train", streaming=True)
    ds = ds.cast_column("audio_filepath", Audio(sampling_rate=TARGET_SAMPLE_RATE))

    counts = {k: 0 for k in BUCKETS}
    metadata = []

    # Using enumerate to track progress in the terminal
    for i, example in enumerate(tqdm(ds, desc="Sourcing Audio")):

        # 3. Filter Case-Insensitive
        scenario = example.get("scenario", "").lower()
        if scenario not in ["extempore", "conversation", "conversational"]:
            continue

        # 4. Access the Audio Dictionary
        audio_data = example.get("audio_filepath")
        if not audio_data:
            continue

        y = audio_data["array"]
        sr = audio_data["sampling_rate"]
        duration = len(y) / sr

        # 5. Assign to Bucket
        assigned_bucket = None
        for name, (min_d, max_d) in BUCKETS.items():
            if min_d <= duration <= max_d and counts[name] < CLIPS_PER_BUCKET:
                assigned_bucket = name
                break

        if not assigned_bucket:
            continue

        # 6. Save the File
        file_name = f"clip_{counts[assigned_bucket]}.wav"
        file_path = os.path.join(OUT_DIR, assigned_bucket, file_name)

        sf.write(file_path, y, TARGET_SAMPLE_RATE)

        # 7. Use the CLEAN 'normalized' text for the Ground Truth
        metadata.append({
            "file": file_path,
            "duration": round(duration, 2),
            "bucket": assigned_bucket,
            "text": example.get("normalized", "")
        })

        counts[assigned_bucket] += 1

        # Stop if we are finished
        if all(c >= CLIPS_PER_BUCKET for c in counts.values()):
            break

    # 8. Save the Reference Sheet
    df = pd.DataFrame(metadata)
    df.to_csv(os.path.join(OUT_DIR, "metadata.csv"), index=False)
    print("\n✅ Dataset Sourced! Check the 'benchmark_data' folder.")

if __name__ == "__main__":
    MY_TOKEN = "Your_hf_Token"
    prepare_folders()
    get_audio_data(MY_TOKEN)